# Colab 5: Text Generative Perplexity (Figure 3)
## Reproducing Figure 3 from "Train for the Worst, Plan for the Best" (Kim et al., ICML 2025)

**Experiment**: Compare vanilla vs adaptive MDM inference on text generation,
measuring Generative Perplexity (GenPPL) and unigram Entropy across different sampling step counts.

**Figure 3 details**:
- x-axis: Sampling Steps (250, 500, 750, 1000, 1250, 1500, 1750, 2000)
- Left y-axis: Generative Perplexity (GenPPL)
- Right y-axis: Entropy
- Lines: GenPPL vanilla, GenPPL adaptive, Entropy vanilla, Entropy adaptive

**Model**: 1.1B MDM (hidden=2048, layers=24, heads=32, ff=8192) pretrained on text

**Evaluator**: LLaMA-2 7B base model

**Adaptive oracle**: `top_prob_margin` with Gaussian noise (NOT Gumbel)
- F = Top K(margin + epsilon), epsilon ~ N(0, sigma^2)
- K = (# masked) x (alpha_s - alpha_t) / (1 - alpha_t), linear schedule alpha_t = 1-t

**Runtime**: A100 GPU (80 GB), ~12-24 hours total

---
## Cell 1: Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/mdm_reproduction'

!pip install torch torchvision transformers einops tqdm matplotlib accelerate sentencepiece protobuf -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
import math
import gc
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

os.makedirs(f'{BASE_DIR}/figures', exist_ok=True)
os.makedirs(f'{BASE_DIR}/results', exist_ok=True)
os.makedirs(f'{BASE_DIR}/models', exist_ok=True)
os.makedirs(f'{BASE_DIR}/data/text', exist_ok=True)

---
## Cell 2: Clone SMDM Codebase (Nie et al. 2024)

The pretrained 1.1B MDM checkpoint comes from the SMDM (Simplified Masked Diffusion Models)
codebase. This is the same checkpoint referenced in Kim et al. (Appendix D.1.2):
"We use pretrained MDMs on text from Nie et al. (2024)."

The SMDM repo provides:
- Pretrained 1.1B MDM checkpoint (trained on SlimPajama)
- Tokenizer (same as LLaMA)
- Model architecture definition
- Mask token convention

In [ ]:
SMDM_DIR = '/content/smdm'

if not os.path.exists(SMDM_DIR):
    !git clone https://github.com/smdm-anonymous/smdm.git {SMDM_DIR}
else:
    print(f"SMDM repo already cloned at {SMDM_DIR}")

# List contents to understand structure
!ls {SMDM_DIR}/
print()
!ls {SMDM_DIR}/*.py 2>/dev/null || echo 'No .py files at top level'
print()
# Check for model definitions
!find {SMDM_DIR} -name '*.py' -maxdepth 3 | head -20

---
## Cell 3: 1.1B MDM Architecture

The 1.1B MDM follows the architecture from MDLM/SMDM:
- Bidirectional transformer (no causal mask)
- No time embedding (time-embedding-free, as in Zheng et al. 2024)
- RoPE positional embeddings
- Config: hidden=2048, layers=24, heads=32, ff_dim=8192
- Vocabulary: 32000 (LLaMA tokenizer) + 1 mask token = 32001
- Max sequence length: 1024

If the SMDM codebase provides a model class, we use it directly.
Otherwise, we define a compatible architecture below.

In [ ]:
import sys
sys.path.insert(0, SMDM_DIR)

# Attempt to import SMDM model definition
try:
    # SMDM typically uses a DiT-style or GPT-style backbone
    from model import MDMModel  # Adjust import path based on repo structure
    print("Loaded MDMModel from SMDM codebase.")
    USE_SMDM_MODEL = True
except ImportError:
    print("Could not import from SMDM. Defining compatible architecture.")
    USE_SMDM_MODEL = False

# ---- Fallback: Define 1.1B MDM from scratch ----
class RoPE(nn.Module):
    """Rotary Position Embedding."""
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self.max_seq_len = max_seq_len

    def forward(self, x, seq_len):
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]


def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, ff_dim, bias=False),
            nn.GELU(),
            nn.Linear(ff_dim, hidden_dim, bias=False),
        )
        if dropout > 0:
            self.attn_dropout = nn.Dropout(dropout)
        else:
            self.attn_dropout = nn.Identity()

    def forward(self, x, rope_cos=None, rope_sin=None):
        B, L, D = x.shape
        h = self.ln1(x)
        qkv = self.qkv(h).reshape(B, L, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, L, D_h)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if rope_cos is not None:
            q, k = apply_rotary_pos_emb(q, k, rope_cos, rope_sin)
        # Use scaled_dot_product_attention for memory efficiency
        out = F.scaled_dot_product_attention(q, k, v)
        out = out.transpose(1, 2).reshape(B, L, D)
        out = self.out_proj(out)
        x = x + out
        return x + self.ff(self.ln2(x))


class TextMDM(nn.Module):
    """
    1.1B Masked Diffusion Model for text.
    Bidirectional attention, no time embedding, RoPE.
    Vocabulary: 32000 (LLaMA) + 1 mask token.
    """
    def __init__(self, vocab_size=32001, hidden_dim=2048, num_layers=24,
                 num_heads=32, ff_dim=8192, max_seq_len=1024, dropout=0.0):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.max_seq_len = max_seq_len
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.rope = RoPE(hidden_dim // num_heads, max_seq_len)
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.ln_final = nn.LayerNorm(hidden_dim)
        self.output_head = nn.Linear(hidden_dim, vocab_size, bias=False)

    def forward(self, x):
        B, L = x.shape
        h = self.embedding(x)
        rope_cos, rope_sin = self.rope(h, L)
        for block in self.blocks:
            h = block(h, rope_cos, rope_sin)
        h = self.ln_final(h)
        return self.output_head(h)


# Config for 1.1B model
MDM_CONFIG = {
    'vocab_size': 32001,     # 32000 LLaMA tokens + 1 mask token
    'hidden_dim': 2048,
    'num_layers': 24,
    'num_heads': 32,
    'ff_dim': 8192,
    'max_seq_len': 1024,
}

# Mask token is the last token in the vocabulary
MASK_TOKEN_ID = 32000  # index 32000 (0-indexed)
VOCAB_SIZE = 32001
SEQ_LEN = 1024

# Verify parameter count
test_model = TextMDM(**MDM_CONFIG)
num_params = sum(p.numel() for p in test_model.parameters())
print(f"TextMDM parameters: {num_params / 1e9:.2f}B ({num_params / 1e6:.0f}M)")
del test_model
torch.cuda.empty_cache()

---
## Cell 4: Load Pretrained 1.1B MDM Checkpoint

The SMDM codebase provides pretrained checkpoints.
If unavailable, we download from their release or train from scratch on SlimPajama.

**Checkpoint sources** (in order of preference):
1. SMDM HuggingFace Hub (e.g., `kuleshov-group/mdlm-owt`)
2. Local checkpoint from previous training
3. Train from scratch (requires significant compute)

In [ ]:
from transformers import AutoTokenizer

# ---- Load tokenizer ----
# SMDM / MDLM uses the LLaMA tokenizer (32000 tokens)
TOKENIZER_NAME = 'meta-llama/Llama-2-7b-hf'
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Mask token ID: {MASK_TOKEN_ID}")

# ---- Load or download pretrained MDM ----
checkpoint_path = f'{BASE_DIR}/models/mdm_1.1B_text.pt'

mdm_model = TextMDM(**MDM_CONFIG)

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}")
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    mdm_model.load_state_dict(state_dict, strict=False)
    print("Checkpoint loaded successfully.")
else:
    print(f"No checkpoint found at {checkpoint_path}")
    print("Attempting to load from SMDM / MDLM HuggingFace release...")
    try:
        # MDLM checkpoints on HuggingFace
        from huggingface_hub import hf_hub_download
        # Try known MDLM model IDs
        HF_REPO = 'kuleshov-group/mdlm-owt'  # OpenWebText pretrained
        ckpt_file = hf_hub_download(repo_id=HF_REPO, filename='pytorch_model.bin')
        state_dict = torch.load(ckpt_file, map_location='cpu')
        # Map keys if needed (MDLM may use different naming)
        mdm_model.load_state_dict(state_dict, strict=False)
        # Save locally for next time
        torch.save(mdm_model.state_dict(), checkpoint_path)
        print(f"Loaded from HuggingFace and saved to {checkpoint_path}")
    except Exception as e:
        print(f"HuggingFace download failed: {e}")
        print("\n=== MANUAL CHECKPOINT SETUP REQUIRED ===")
        print("Option 1: Download SMDM pretrained checkpoint and place at:")
        print(f"  {checkpoint_path}")
        print("Option 2: Train from scratch (Cell 4b below)")
        print("Option 3: Use a smaller MDLM checkpoint for rapid prototyping")

mdm_model = mdm_model.to(device).half()  # Use FP16 to fit in GPU memory
mdm_model.eval()
print(f"\nModel on {device}, dtype={next(mdm_model.parameters()).dtype}")
if device == 'cuda':
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## Cell 4b (Optional): Train 1.1B MDM on SlimPajama

Only run this cell if no pretrained checkpoint is available.
Training 1.1B parameters requires multiple A100 GPUs and substantial time.
For rapid prototyping, consider using the MDLM pretrained checkpoint.

In [ ]:
TRAIN_FROM_SCRATCH = False  # Set True only if no checkpoint available

if TRAIN_FROM_SCRATCH:
    from datasets import load_dataset

    print("Loading SlimPajama dataset...")
    # SlimPajama-627B: https://huggingface.co/datasets/cerebras/SlimPajama-627B
    ds = load_dataset('cerebras/SlimPajama-627B', split='train', streaming=True)

    def tokenize_and_chunk(examples):
        """Tokenize and chunk text into fixed-length sequences."""
        tokens = tokenizer(
            examples['text'], truncation=False, padding=False,
            return_attention_mask=False
        )['input_ids']
        # Concatenate all tokens and split into chunks of SEQ_LEN
        all_tokens = [t for seq in tokens for t in seq]
        chunks = [all_tokens[i:i + SEQ_LEN]
                  for i in range(0, len(all_tokens) - SEQ_LEN + 1, SEQ_LEN)]
        return {'input_ids': chunks}

    # MDM training step
    def mdm_text_train_step(model, optimizer, input_ids, mask_token_id):
        model.train()
        B, L = input_ids.shape
        n = torch.randint(1, L + 1, (B,), device=input_ids.device)
        noise = torch.rand(B, L, device=input_ids.device)
        sorted_idx = noise.argsort(dim=-1)
        mask = torch.zeros(B, L, dtype=torch.bool, device=input_ids.device)
        for b in range(B):
            mask[b, sorted_idx[b, :n[b]]] = True
        x_masked = input_ids.clone()
        x_masked[mask] = mask_token_id
        logits = model(x_masked)
        total_loss = torch.tensor(0.0, device=input_ids.device)
        for b in range(B):
            if mask[b].sum() > 0:
                sample_loss = F.cross_entropy(
                    logits[b, mask[b]], input_ids[b, mask[b]], reduction='sum'
                )
                total_loss = total_loss + sample_loss / n[b].float()
        total_loss = total_loss / B
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        return total_loss.item()

    # Training loop (abbreviated — full training requires days on multi-GPU)
    mdm_model = mdm_model.float()  # Need FP32 for training
    optimizer = torch.optim.AdamW(
        mdm_model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1
    )

    NUM_TRAIN_STEPS = 100000
    BATCH_SIZE = 4  # Small for single GPU
    buffer = []
    step = 0

    for example in tqdm(ds, total=NUM_TRAIN_STEPS * BATCH_SIZE, desc="Training"):
        tokens = tokenizer(example['text'], truncation=True, max_length=SEQ_LEN,
                           padding='max_length', return_tensors='pt')['input_ids'][0]
        buffer.append(tokens)
        if len(buffer) >= BATCH_SIZE:
            batch = torch.stack(buffer[:BATCH_SIZE]).to(device)
            buffer = buffer[BATCH_SIZE:]
            loss = mdm_text_train_step(mdm_model, optimizer, batch, MASK_TOKEN_ID)
            step += 1
            if step % 500 == 0:
                print(f"Step {step}, Loss: {loss:.4f}")
            if step >= NUM_TRAIN_STEPS:
                break

    torch.save(mdm_model.state_dict(), checkpoint_path)
    print(f"Training complete. Saved to {checkpoint_path}")
    mdm_model = mdm_model.half()
    mdm_model.eval()
else:
    print("Skipping training — using pretrained checkpoint.")

---
## Cell 5: MDM Inference — Vanilla and Adaptive

**Vanilla**: random order of unmasking (baseline).

**Adaptive (top_prob_margin with Gaussian noise)**:
- Certainty score = |p(j1) - p(j2)| (margin between top-2 predicted tokens)
- Add Gaussian noise: F = Top K(margin + epsilon), epsilon ~ N(0, sigma^2)
- sigma = 0.001 (Appendix D.1.2, text oracle with Gaussian noise)

**Schedule**:
- alpha_t = 1 - t (linear), reverse from t=1 to t=0
- K = (# masked) x (alpha_s - alpha_t) / (1 - alpha_t)
  = (# masked) x (t - s) / t

In [ ]:
GAUSSIAN_SIGMA = 0.001  # Appendix D.1.2: Gaussian noise for text oracle


@torch.no_grad()
def mdm_text_inference(model, seq_len, mask_token_id, vocab_size,
                       strategy='vanilla', num_steps=1000,
                       gaussian_sigma=0.001, device='cuda',
                       batch_size=1):
    """
    MDM reverse-process inference for text generation.

    Args:
        strategy: 'vanilla' or 'adaptive' (top_prob_margin + Gaussian)
        num_steps: number of reverse sampling steps
        gaussian_sigma: std dev of Gaussian noise for adaptive strategy
        batch_size: number of sequences to generate in parallel

    Returns:
        (batch_size, seq_len) tensor of generated token IDs
    """
    model.eval()
    B = batch_size

    # Initialize fully masked
    x = torch.full((B, seq_len), mask_token_id, dtype=torch.long, device=device)

    # Linear noise schedule: alpha_t = 1 - t, reverse from t=1 to t=0
    timesteps = torch.linspace(1.0, 0.0, num_steps + 1)

    for step in range(num_steps):
        t = timesteps[step].item()
        s = timesteps[step + 1].item()

        # Identify masked positions per sequence
        is_masked = (x == mask_token_id)  # (B, seq_len)
        num_masked_per_seq = is_masked.sum(dim=-1)  # (B,)

        # If no sequence has any masked tokens, we are done
        if num_masked_per_seq.max().item() == 0:
            break

        # Forward pass (batched)
        logits = model(x)  # (B, seq_len, vocab_size)

        # Exclude mask token from predictions
        logits[:, :, mask_token_id] = float('-inf')
        probs = F.softmax(logits, dim=-1)  # (B, seq_len, vocab_size)

        for b in range(B):
            masked_positions = is_masked[b].nonzero(as_tuple=True)[0]
            num_masked = len(masked_positions)
            if num_masked == 0:
                continue

            # K = num_masked * (t - s) / t
            unmask_frac = (t - s) / (t + 1e-10)
            K = max(1, round(num_masked * unmask_frac))
            K = min(K, num_masked)

            masked_probs = probs[b, masked_positions]  # (num_masked, vocab_size)

            if strategy == 'vanilla':
                # Random selection of K positions to unmask
                perm = torch.randperm(num_masked, device=device)[:K]
                selected_positions = masked_positions[perm]
            elif strategy == 'adaptive':
                # Top-prob-margin with Gaussian noise
                top2_vals, _ = torch.topk(masked_probs,
                                          k=min(2, masked_probs.shape[-1]), dim=-1)
                if top2_vals.shape[-1] >= 2:
                    margin = top2_vals[:, 0] - top2_vals[:, 1]
                else:
                    margin = top2_vals[:, 0]
                # Add Gaussian noise: F = margin + epsilon, epsilon ~ N(0, sigma^2)
                epsilon = torch.randn_like(margin) * gaussian_sigma
                scores = margin + epsilon
                _, top_k_idx = torch.topk(scores, K)
                selected_positions = masked_positions[top_k_idx]
            else:
                raise ValueError(f"Unknown strategy: {strategy}")

            # Sample tokens for selected positions
            for pos in selected_positions:
                token = torch.multinomial(probs[b, pos], 1).item()
                x[b, pos] = token

    # Fill any remaining masked positions greedily
    remaining = (x == mask_token_id)
    if remaining.any():
        logits = model(x)
        logits[:, :, mask_token_id] = float('-inf')
        for b in range(B):
            rem_pos = remaining[b].nonzero(as_tuple=True)[0]
            for pos in rem_pos:
                x[b, pos] = torch.multinomial(
                    F.softmax(logits[b, pos], dim=-1), 1
                ).item()

    return x


print("Inference functions defined.")
print(f"Gaussian sigma for text oracle: {GAUSSIAN_SIGMA}")

---
## Cell 6: Load LLaMA-2 7B Evaluator

GenPPL is computed using LLaMA-2 7B BASE (not chat) as the evaluator:

GenPPL = exp(-1/N * sum_{i=1}^{N} log p_eval(x_i | x_{<i}))

where p_eval is the autoregressive probability from LLaMA-2 7B.

In [ ]:
from transformers import AutoModelForCausalLM

LLAMA_MODEL_NAME = 'meta-llama/Llama-2-7b-hf'

print("Loading LLaMA-2 7B evaluator...")
print("NOTE: This requires access to the LLaMA-2 model on HuggingFace.")
print("Run `huggingface-cli login` with an approved token first.")
print()

# Free MDM model memory temporarily if needed
# mdm_model = mdm_model.cpu(); torch.cuda.empty_cache()

llama_model = AutoModelForCausalLM.from_pretrained(
    LLAMA_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',  # Automatic device placement for large model
    low_cpu_mem_usage=True,
)
llama_model.eval()

llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA_MODEL_NAME, use_fast=True)
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

print(f"LLaMA-2 7B loaded.")
if device == 'cuda':
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## Cell 7: Metric Computation — GenPPL and Entropy

**GenPPL**: exp(-1/N * sum log p_eval(x_i | x_{<i}))
- Computed using LLaMA-2 7B as autoregressive evaluator
- Average over all tokens in all generated sequences

**Entropy**: -sum p_i log p_i
- Unigram frequency: p_i = count(token_i) / total tokens
- Computed over all generated tokens across the batch

In [ ]:
@torch.no_grad()
def compute_genppl(generated_ids, eval_model, chunk_size=256):
    """
    Compute Generative Perplexity using an autoregressive evaluator.

    GenPPL = exp( -1/N * sum_{i=1}^{N} log p_eval(x_i | x_{<i}) )

    Args:
        generated_ids: (num_samples, seq_len) tensor of token IDs
        eval_model: LLaMA-2 7B model
        chunk_size: process this many tokens at a time to avoid OOM

    Returns:
        genppl: float
    """
    eval_model.eval()
    total_log_prob = 0.0
    total_tokens = 0

    for seq_idx in range(generated_ids.shape[0]):
        seq = generated_ids[seq_idx].unsqueeze(0)  # (1, seq_len)
        seq_len = seq.shape[1]

        # Process in chunks to manage memory
        # For each token position i, we need log p(x_i | x_{<i})
        # We can get this from a single forward pass: logits[:, :-1] predict tokens[:, 1:]
        if seq_len <= chunk_size:
            logits = eval_model(seq.to(eval_model.device)).logits  # (1, seq_len, vocab)
            # logits[:, i] predicts token at position i+1
            shift_logits = logits[:, :-1, :]  # (1, seq_len-1, vocab)
            shift_labels = seq[:, 1:].to(eval_model.device)  # (1, seq_len-1)
            log_probs = F.log_softmax(shift_logits, dim=-1)
            token_log_probs = log_probs.gather(
                dim=-1, index=shift_labels.unsqueeze(-1)
            ).squeeze(-1)  # (1, seq_len-1)
            total_log_prob += token_log_probs.sum().item()
            total_tokens += seq_len - 1
        else:
            # Sliding window for very long sequences
            for start in range(0, seq_len - 1, chunk_size):
                end = min(start + chunk_size, seq_len)
                # Include one extra token at the start for context
                ctx_start = max(0, start - chunk_size)
                chunk = seq[:, ctx_start:end].to(eval_model.device)
                logits = eval_model(chunk).logits
                # Only score the tokens in [start, end-1]
                offset = start - ctx_start
                if offset > 0:
                    score_logits = logits[:, offset - 1:-1, :]
                    score_labels = chunk[:, offset:]
                else:
                    score_logits = logits[:, :-1, :]
                    score_labels = chunk[:, 1:]
                log_probs = F.log_softmax(score_logits, dim=-1)
                token_log_probs = log_probs.gather(
                    dim=-1, index=score_labels.unsqueeze(-1)
                ).squeeze(-1)
                total_log_prob += token_log_probs.sum().item()
                total_tokens += score_labels.shape[1]

    avg_neg_log_prob = -total_log_prob / max(total_tokens, 1)
    genppl = math.exp(avg_neg_log_prob)
    return genppl


def compute_unigram_entropy(generated_ids, vocab_size=32000):
    """
    Compute unigram entropy of generated text.

    Entropy = -sum_i p_i * log(p_i)
    where p_i = count(token_i) / total_tokens

    Args:
        generated_ids: (num_samples, seq_len) tensor or numpy array
        vocab_size: size of token vocabulary (excluding mask)

    Returns:
        entropy: float (in nats)
    """
    if isinstance(generated_ids, torch.Tensor):
        flat_tokens = generated_ids.cpu().numpy().flatten()
    else:
        flat_tokens = generated_ids.flatten()

    # Count token frequencies
    counts = np.bincount(flat_tokens, minlength=vocab_size)
    # Only consider non-mask tokens
    counts = counts[:vocab_size]  # Exclude mask token if present
    total = counts.sum()

    if total == 0:
        return 0.0

    # p_i = count_i / total
    probs = counts / total
    # Entropy = -sum p_i * log(p_i), skip zero-count tokens
    nonzero = probs > 0
    entropy = -np.sum(probs[nonzero] * np.log(probs[nonzero]))

    return entropy


print("Metric functions defined.")

---
## Cell 8: Generate Text and Evaluate at Each Step Count

For each number of sampling steps (250, 500, ..., 2000):
1. Generate a batch of text sequences with vanilla MDM inference
2. Generate a batch of text sequences with adaptive MDM inference
3. Compute GenPPL and Entropy for each batch

Batch size per step count: 64 sequences (balances compute and statistical reliability)

In [ ]:
STEP_COUNTS = [250, 500, 750, 1000, 1250, 1500, 1750, 2000]
NUM_SAMPLES = 64       # Sequences per (strategy, step_count) combination
GEN_BATCH_SIZE = 4     # Batch size for generation (limited by GPU memory for 1.1B model)
EVAL_SEQ_LEN = SEQ_LEN  # 1024 tokens per sequence

results = {
    'step_counts': STEP_COUNTS,
    'genppl_vanilla': [],
    'genppl_adaptive': [],
    'entropy_vanilla': [],
    'entropy_adaptive': [],
}

results_path = f'{BASE_DIR}/results/figure3_text_genppl.json'

# Check for cached results
if os.path.exists(results_path):
    with open(results_path) as f:
        cached = json.load(f)
    if len(cached.get('genppl_vanilla', [])) == len(STEP_COUNTS):
        print(f"Loaded cached results from {results_path}")
        results = cached
        SKIP_GENERATION = True
    else:
        SKIP_GENERATION = False
else:
    SKIP_GENERATION = False

if not SKIP_GENERATION:
    # Ensure MDM model is on GPU
    mdm_model = mdm_model.to(device)

    for num_steps in STEP_COUNTS:
        print(f"\n{'='*60}")
        print(f"Sampling Steps: {num_steps}")
        print(f"{'='*60}")

        for strategy in ['vanilla', 'adaptive']:
            print(f"  Strategy: {strategy}")
            all_generated = []

            num_batches = (NUM_SAMPLES + GEN_BATCH_SIZE - 1) // GEN_BATCH_SIZE
            for batch_idx in tqdm(range(num_batches), desc=f"    Generating"):
                bs = min(GEN_BATCH_SIZE, NUM_SAMPLES - batch_idx * GEN_BATCH_SIZE)
                generated = mdm_text_inference(
                    mdm_model, seq_len=EVAL_SEQ_LEN,
                    mask_token_id=MASK_TOKEN_ID, vocab_size=VOCAB_SIZE,
                    strategy=strategy, num_steps=num_steps,
                    gaussian_sigma=GAUSSIAN_SIGMA, device=device,
                    batch_size=bs
                )
                all_generated.append(generated.cpu())

            all_generated = torch.cat(all_generated, dim=0)  # (NUM_SAMPLES, seq_len)
            print(f"    Generated {all_generated.shape[0]} sequences of length {all_generated.shape[1]}")

            # Save generated samples
            save_dir = f'{BASE_DIR}/data/text/fig3_samples'
            os.makedirs(save_dir, exist_ok=True)
            torch.save(all_generated, f'{save_dir}/{strategy}_steps{num_steps}.pt')

            # Compute entropy (fast, CPU-only)
            entropy = compute_unigram_entropy(all_generated, vocab_size=VOCAB_SIZE - 1)
            print(f"    Entropy: {entropy:.4f}")

            # Compute GenPPL using LLaMA-2 7B
            # Move MDM off GPU if needed to make room for LLaMA
            mdm_on_gpu = True
            if device == 'cuda':
                free_mem = torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()
                if free_mem < 15e9:  # Need ~14GB for LLaMA-2 7B in FP16
                    print("    Moving MDM to CPU to make room for LLaMA evaluator...")
                    mdm_model = mdm_model.cpu()
                    torch.cuda.empty_cache()
                    mdm_on_gpu = False

            genppl = compute_genppl(all_generated, llama_model)
            print(f"    GenPPL: {genppl:.2f}")

            # Move MDM back to GPU if it was moved
            if not mdm_on_gpu:
                mdm_model = mdm_model.to(device)

            # Store results
            results[f'genppl_{strategy}'].append(genppl)
            results[f'entropy_{strategy}'].append(entropy)

        # Save intermediate results after each step count
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"  Intermediate results saved.")

    print(f"\nAll results saved to {results_path}")

# Print summary
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"{'Steps':>6}  {'GenPPL Van':>12}  {'GenPPL Adpt':>12}  {'Ent Van':>10}  {'Ent Adpt':>10}")
print("-" * 60)
for i, steps in enumerate(STEP_COUNTS):
    gv = results['genppl_vanilla'][i] if i < len(results['genppl_vanilla']) else float('nan')
    ga = results['genppl_adaptive'][i] if i < len(results['genppl_adaptive']) else float('nan')
    ev = results['entropy_vanilla'][i] if i < len(results['entropy_vanilla']) else float('nan')
    ea = results['entropy_adaptive'][i] if i < len(results['entropy_adaptive']) else float('nan')
    print(f"{steps:>6}  {gv:>12.2f}  {ga:>12.2f}  {ev:>10.4f}  {ea:>10.4f}")

---
## Cell 9: Decode and Inspect Generated Text

Sanity check: decode a few generated sequences to verify they look like coherent text.

In [ ]:
# Load a sample batch for inspection
sample_file = f'{BASE_DIR}/data/text/fig3_samples/adaptive_steps2000.pt'
if os.path.exists(sample_file):
    samples = torch.load(sample_file, map_location='cpu')
    print(f"Loaded {samples.shape[0]} samples from adaptive/2000 steps")
    print()
    for i in range(min(3, samples.shape[0])):
        text = tokenizer.decode(samples[i], skip_special_tokens=True)
        print(f"--- Sample {i+1} (first 300 chars) ---")
        print(text[:300])
        print()
else:
    print(f"No samples found at {sample_file}")
    print("Run Cell 8 first to generate samples.")

---
## Cell 10: Plot Figure 3 — Generative Perplexity Comparison

Dual y-axis plot:
- Left y-axis: Generative Perplexity (GenPPL)
- Right y-axis: Entropy
- x-axis: Sampling Steps
- Four lines: GenPPL vanilla, GenPPL adaptive, Entropy vanilla, Entropy adaptive

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Publication-quality settings
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.size'] = 12
matplotlib.rcParams['axes.linewidth'] = 1.0
matplotlib.rcParams['xtick.major.width'] = 1.0
matplotlib.rcParams['ytick.major.width'] = 1.0
matplotlib.rcParams['xtick.direction'] = 'in'
matplotlib.rcParams['ytick.direction'] = 'in'
matplotlib.rcParams['legend.frameon'] = True
matplotlib.rcParams['legend.edgecolor'] = 'black'
matplotlib.rcParams['legend.fancybox'] = False

# Load results
with open(f'{BASE_DIR}/results/figure3_text_genppl.json') as f:
    results = json.load(f)

steps = results['step_counts']
genppl_vanilla = results['genppl_vanilla']
genppl_adaptive = results['genppl_adaptive']
entropy_vanilla = results['entropy_vanilla']
entropy_adaptive = results['entropy_adaptive']

# ---- Create Figure ----
fig, ax1 = plt.subplots(figsize=(7, 4.5))

# Left y-axis: GenPPL
color_vanilla_genppl = '#2166ac'    # Blue
color_adaptive_genppl = '#b2182b'   # Red
color_vanilla_entropy = '#4393c3'   # Light blue
color_adaptive_entropy = '#d6604d'  # Light red

line1, = ax1.plot(steps, genppl_vanilla, 'o-', color=color_vanilla_genppl,
                  linewidth=2, markersize=6, label='GenPPL vanilla')
line2, = ax1.plot(steps, genppl_adaptive, 's-', color=color_adaptive_genppl,
                  linewidth=2, markersize=6, label='GenPPL adaptive')

ax1.set_xlabel('Sampling Steps', fontsize=13)
ax1.set_ylabel('Generative Perplexity', fontsize=13, color='black')
ax1.tick_params(axis='y', labelcolor='black')
ax1.set_xticks(steps)
ax1.set_xticklabels([str(s) for s in steps], fontsize=10)

# Right y-axis: Entropy
ax2 = ax1.twinx()
line3, = ax2.plot(steps, entropy_vanilla, '^--', color=color_vanilla_entropy,
                  linewidth=2, markersize=6, label='Entropy vanilla')
line4, = ax2.plot(steps, entropy_adaptive, 'v--', color=color_adaptive_entropy,
                  linewidth=2, markersize=6, label='Entropy adaptive')

ax2.set_ylabel('Entropy', fontsize=13, color='black')
ax2.tick_params(axis='y', labelcolor='black')

# Combined legend
lines = [line1, line2, line3, line4]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right', fontsize=10, framealpha=0.9)

ax1.set_title('Figure 3: Text GenPPL — Vanilla vs Adaptive MDM', fontsize=14)
ax1.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()

# Save
fig_dir = f'{BASE_DIR}/figures'
os.makedirs(fig_dir, exist_ok=True)
plt.savefig(f'{fig_dir}/figure3_text_genppl.pdf', dpi=300, bbox_inches='tight')
plt.savefig(f'{fig_dir}/figure3_text_genppl.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure saved to {fig_dir}/figure3_text_genppl.pdf")
print(f"Figure saved to {fig_dir}/figure3_text_genppl.png")

---
## Cell 11: Results Summary and Comparison with Paper

**Expected trends from Figure 3 (Kim et al., 2025)**:
- GenPPL decreases as sampling steps increase (more steps = better quality)
- Adaptive GenPPL is consistently LOWER than vanilla GenPPL (adaptive is better)
- Entropy for adaptive is slightly higher than vanilla (more diverse text)
- The gap between vanilla and adaptive is most pronounced at fewer steps

In [ ]:
print("=" * 70)
print("FIGURE 3 RESULTS: Text Generative Perplexity")
print("=" * 70)
print()
print(f"{'Steps':>6}  {'GenPPL Van':>12}  {'GenPPL Adpt':>12}  {'Ent Van':>10}  {'Ent Adpt':>10}")
print("-" * 60)
for i, s in enumerate(results['step_counts']):
    gv = results['genppl_vanilla'][i]
    ga = results['genppl_adaptive'][i]
    ev = results['entropy_vanilla'][i]
    ea = results['entropy_adaptive'][i]
    print(f"{s:>6}  {gv:>12.2f}  {ga:>12.2f}  {ev:>10.4f}  {ea:>10.4f}")

print()
print("Key observations:")
avg_van = np.mean(results['genppl_vanilla'])
avg_adp = np.mean(results['genppl_adaptive'])
improvement = (avg_van - avg_adp) / avg_van * 100
print(f"  Average GenPPL vanilla:  {avg_van:.2f}")
print(f"  Average GenPPL adaptive: {avg_adp:.2f}")
print(f"  Average improvement:     {improvement:.1f}%")
print()

avg_ent_van = np.mean(results['entropy_vanilla'])
avg_ent_adp = np.mean(results['entropy_adaptive'])
print(f"  Average Entropy vanilla:  {avg_ent_van:.4f}")
print(f"  Average Entropy adaptive: {avg_ent_adp:.4f}")
print()
print("Expected: Adaptive should have LOWER GenPPL (better text quality)")
print("          and comparable or slightly higher Entropy (diversity).")
print()
print("Paper reference: Kim et al. (2025), Figure 3")
print("  MDM: 1.1B params, Evaluator: LLaMA-2 7B")
print("  Adaptive oracle: top_prob_margin with Gaussian noise (sigma=0.001)")